In [1]:
import os, sys
import re
import csv
import argparse
from pathlib import Path

sys.path.append('/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM')

from src.preprocess import (strip_pdb,
                            create_dataframe, 
                            extract_chains_chothia,
                            fix_case_if_reused,
                            build_interface_pair)

In [5]:
BASE_DIR = Path("/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data")
pdb = "8jiu.pdb"

pdbPath = BASE_DIR / pdb

In [26]:
def get_interface_chains(pdbPath):
    #extract chains
    try:
        chain_pairs = extract_chains_chothia(pdbPath)
        if not chain_pairs:
            raise ValueError(f"No chain pairs returned for {pdbPath}")
        ligand_chains, receptor_chains = chain_pairs[0]


        ### Some chains are annotated with int like "2" in sabdab. This leads to unphysical brooms downstream
        if not all(isinstance(v, str)and  v.isalpha() for v in ligand_chains):
            raise ValueError(f"Invalid characters in ligand chains: {ligand_chains}")

        if not all(isinstance(v, str)and  v.isalpha() for v in receptor_chains):
            raise ValueError(f"Invalid characters in receptor chains: {receptor_chains}")

        receptor_chains = fix_case_if_reused(ligand_chains, receptor_chains, Path(pdbPath).stem) # some chothia pdbs have wrong headers


    except Exception as e:
        print(f"[ERROR] Problem processing file: {pdbPath}")
        print(f"Reason: {e}")
        exit(1)  # Non-zero exit code so subprocess.run will detect failure
    return ligand_chains + receptor_chains

In [7]:

def parse_pocket_file(pocket_path):
    """
    Parse an fpocket pocket*_atm.pdb file.

    Returns:
        header: dict of parsed header values, with snake_case keys
        chains: set of chain identifiers observed in ATOM lines
    """
    header = {}
    chains = set()
    header_pattern = re.compile(r"HEADER\s+\d+\s*-\s*(.*?)\s*:\s*([\d\.]+)")
    with open(pocket_path) as f:
        for line in f:
            if line.startswith('HEADER'):
                m = header_pattern.search(line)
                if m:
                    name, value = m.groups()
                    # convert name to snake_case key
                    key = re.sub(r"[^0-9a-zA-Z]+", "_", name.strip()).lower()
                    # try int, then float
                    if value.isdigit():
                        header[key] = int(value)
                    else:
                        header[key] = float(value)
            elif line.startswith(('ATOM', 'HETATM')):
                parts = line.split()
                if len(parts) >= 5:
                    chain = parts[4]
                    chains.add(chain)
    return header, chains



In [8]:
test_path = Path("/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/negativeSamples/fpocket/output/8jiu_inter_out/pockets/pocket1_atm.pdb")


In [9]:
h,c = parse_pocket_file(test_path)

In [10]:
h

{'pocket_score': 0.256,
 'drug_score': 0.5145,
 'number_of_alpha_spheres': 48,
 'mean_alpha_sphere_radius': 3.9133,
 'mean_alpha_sphere_solvent_acc_': 0.4764,
 'mean_b_factor_of_pocket_residues': 0.5517,
 'hydrophobicity_score': 24.5333,
 'polarity_score': 5,
 'amino_acid_based_volume_score': 4.2,
 'pocket_volume_monte_carlo_': 438.2712,
 'pocket_volume_convex_hull_': 74.9122,
 'charge_score': 1,
 'local_hydrophobic_density_score': 22.5385,
 'number_of_apolar_alpha_sphere': 26,
 'proportion_of_apolar_alpha_sphere': 0.5417}

In [13]:
def filter_pocket_ids(pockets_dir, target_chains, min_chains=2, min_values=None, max_values=None):
    """
    Walk through all pocket*_atm.pdb files in a given pockets/ directory
    and return a list of pocket IDs (e.g. 'pocket0', 'pocket1', …) that:
      1) contain at least `min_chains` distinct chains from `target_chains`
      2) satisfy all numeric header thresholds in min_values / max_values

    Args:
        pockets_dir (str or Path): path to the pockets/ subdirectory
        target_chains (list of str): chains to look for, e.g. ['A','B','F']
        min_chains (int): minimum number of those chains that must appear
        min_values (dict): e.g. {'pocket_volume_monte_carlo': 100, 'hydrophobicity_score': 30}
        max_values (dict): e.g. {'number_of_alpha_spheres': 50}

    Returns:
        List[str]: pocket IDs (filenames without “_atm.pdb”) passing the filters
    """
    pockets_dir = Path(pockets_dir)
    min_values = min_values or {}
    max_values = max_values or {}

    pocket_pattern = re.compile(r'^(pocket\d+)_atm\.pdb$')
    selected = []

    for pocket_file in pockets_dir.iterdir():
        if not pocket_file.is_file():
            continue

        m = pocket_pattern.match(pocket_file.name)
        if not m:
            continue

        pocket_id = m.group(1)

        # parse header & chains from the file
        header, chains = parse_pocket_file(pocket_file)

        # 1) chain adjacency filter
        present = set(target_chains) & chains
        if len(present) < min_chains:
            continue

        # 2) numeric header filters
        if any(header.get(k, float('-inf')) < v for k, v in min_values.items()):
            continue
        if any(header.get(k, float('inf')) > v for k, v in max_values.items()):
            continue

        selected.append(pocket_id)

    return selected

In [24]:
#sel_poc = filter_pocket_ids(test_path.parent, chains,
#                  min_values= { 'hydrophobicity_score': 16},
#                  max_values= {'pocket_volume_monte_carlo_': 600})

sel_poc = filter_pocket_ids(test_path.parent, chains,
                  min_values= { 'proportion_of_apolar_alpha_sphere': 0.8})

In [25]:
h,c = parse_pocket_file(test_path.parent / f"{sel_poc[0]}_atm.pdb")

IndexError: list index out of range

In [23]:
h

{'pocket_score': 0.0958,
 'drug_score': 0.2807,
 'number_of_alpha_spheres': 49,
 'mean_alpha_sphere_radius': 4.0986,
 'mean_alpha_sphere_solvent_acc_': 0.486,
 'mean_b_factor_of_pocket_residues': 0.0484,
 'hydrophobicity_score': 27.6875,
 'polarity_score': 10,
 'amino_acid_based_volume_score': 4.875,
 'pocket_volume_monte_carlo_': 592.0185,
 'pocket_volume_convex_hull_': 102.5327,
 'charge_score': 4,
 'local_hydrophobic_density_score': 19.5385,
 'number_of_apolar_alpha_sphere': 26,
 'proportion_of_apolar_alpha_sphere': 0.5306}

In [ ]:

def scan_pockets(input_dir, min_chains,
                 min_values, max_values, output_csv=None):
    """
    Scan all *_inter_out/pockets/pocket*_atm.pdb files under input_dir,
    filter by chain adjacency and header thresholds.

    Args:
        input_dir (str): base directory containing pdbID_inter_out folders
        target_chains (list of str): list of chains to require
        min_chains (int): minimum number of distinct target chains per pocket
        min_values (dict): minimum thresholds for header keys
        max_values (dict): maximum thresholds for header keys
        output_csv (str): optional path to write results as CSV
    """
    results = []
    # Walk through directories
    for root, dirs, files in os.walk(input_dir):
        # only process pocket*_atm.pdb files

        for fname in files:
            if fname.endswith('_atm.pdb') and fname.startswith('pocket'):

                pdbID =  
                target_chains = get_interface_chains(f"/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data/{pdbID}.pdb")
                print(target_chains)
                pocket_path = os.path.join(root, fname)
                header, chains = parse_pocket_file(pocket_path)
                # count target chains present
                present = set(target_chains) & chains
                if len(present) < min_chains:
                    break
                # apply header filters
                ok = True
                for key, minv in min_values.items():
                    if key not in header or header[key] < minv:
                        ok = False
                        break
                for key, maxv in max_values.items():
                    if key not in header or header[key] > maxv:
                        ok = False
                        break
                if not ok:
                    continue
                # gather result
                pdb_id = os.path.basename(os.path.dirname(os.path.dirname(pocket_path))).replace('_inter_out','')
                result = {
                    'pdb_id': pdb_id,
                    'pocket_file': pocket_path,
                    'chains': ''.join(sorted(present)),
                }
                # include all header values in result
                result.update(header)
                results.append(result)
    # output
    if output_csv:
        keys = list(results[0].keys()) if results else []
        with open(output_csv, 'w', newline='') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=keys)
            writer.writeheader()
            for row in results:
                writer.writerow(row)
        print(f"Wrote {len(results)} entries to {output_csv}")
    else:
        # print to console
        for row in results:
            print(row)
    return results


In [39]:
res = scan_pockets( input_dir = "/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/negativeSamples/fpocket/output",
                    min_chains = 2,
                    min_values= { 'proportion_of_apolar_alpha_sphere': 0.8},
                    max_values = dict(),
                    output_csv=None)

/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data/pocke.pdb
/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data/pocke.pdb
/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data/pocke.pdb
/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data/pocke.pdb
/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data/pocke.pdb
/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data/pocke.pdb
/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data/pocke.pdb
/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data/pocke.pdb
/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data/pocke.pdb
/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data/pocke.pdb
/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data/pocke.pdb
/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data/pocke.pdb
/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/